# V4 + V5 Training Only With Checkpoint Export


---

# V4 Transformer Adaptive Retrieval


## Goal

Train V4 adaptive retrieval and V5 ranker in one Colab runtime, without heavy candidate evaluation.

Flow:

```text
train V4 -> save/export V4 checkpoint chunks
train V5 from V4 checkpoint -> save/export V5 checkpoint chunks
```

Use this when you want to protect checkpoints first and evaluate later.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import importlib.util
import subprocess
import sys

required = ['pyarrow', 'torch', 'pandas', 'numpy']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages are available.')

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math
import random
import time
from typing import Iterable

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('torch:', torch.__version__)

## Checkpoint-Safe Export Utilities

These utilities split large `.pt` checkpoints into smaller files so browser downloads are less likely to hang. Reports are exported separately as small zip files.


In [ ]:
import hashlib
import shutil
from google.colab import files

CHECKPOINT_CHUNK_SIZE_MB = 128
CHECKPOINT_EXPORT_ROOT = Path('/content/checkpoint_exports')


def human_bytes(n: int | float) -> str:
    n = float(n)
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    for unit in units:
        if abs(n) < 1024 or unit == units[-1]:
            return f'{n:.2f} {unit}'
        n /= 1024


def sha256_file(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def split_checkpoint(checkpoint_path: Path, export_dir: Path, chunk_size_mb: int = CHECKPOINT_CHUNK_SIZE_MB) -> list[Path]:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')
    if export_dir.exists():
        shutil.rmtree(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    chunk_size = int(chunk_size_mb * 1024 * 1024)
    parts = []
    total = checkpoint_path.stat().st_size
    print('splitting checkpoint:', checkpoint_path, human_bytes(total), 'chunk_size=', human_bytes(chunk_size))

    with checkpoint_path.open('rb') as src:
        idx = 0
        while True:
            payload = src.read(chunk_size)
            if not payload:
                break
            part_path = export_dir / f'{checkpoint_path.name}.part_{idx:04d}'
            part_path.write_bytes(payload)
            digest = sha256_file(part_path)
            parts.append({'index': idx, 'filename': part_path.name, 'size_bytes': part_path.stat().st_size, 'sha256': digest})
            print(f'part {idx:04d}:', human_bytes(part_path.stat().st_size), digest[:12])
            idx += 1

    manifest = {
        'checkpoint_name': checkpoint_path.name,
        'total_size_bytes': total,
        'chunk_size_mb': chunk_size_mb,
        'num_parts': len(parts),
        'full_sha256': sha256_file(checkpoint_path),
        'parts': parts,
        'created_at_unix': time.time(),
    }
    (export_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    with (export_dir / 'SHA256SUMS.txt').open('w') as f:
        for part in parts:
            f.write(f"{part['sha256']}  {part['filename']}\n")
        f.write(f"{manifest['full_sha256']}  {checkpoint_path.name}\n")
    print('manifest:', export_dir / 'manifest.json')
    return sorted(export_dir.glob('*.part_*')) + [export_dir / 'manifest.json', export_dir / 'SHA256SUMS.txt']


def export_report_bundle(stage_name: str, artifact_dirs: list[Path]) -> Path | None:
    bundle_dir = Path('/content') / f'{stage_name}_report_bundle'
    if bundle_dir.exists():
        shutil.rmtree(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)

    report_names = [
        'metrics.json', 'config.json', 'training_history.csv', 'candidate_validation.json',
        'ranking_comparison.csv', 'online_model_comparison.json', 'online_replay_summary.json',
        'online_replay_rows.csv', 'v4_candidate_training_manifest.json',
    ]
    copied = []
    for root in artifact_dirs:
        for name in report_names:
            src = root / name
            if src.exists():
                dst = bundle_dir / f'{root.name}_{name}'
                shutil.copy2(src, dst)
                copied.append(dst)
                print('copied report:', src, '->', dst)
    if not copied:
        print('No report files found for', stage_name)
        return None
    zip_path = Path(shutil.make_archive(str(Path('/content') / f'{stage_name}_report_bundle'), 'zip', str(bundle_dir)))
    print('report bundle:', zip_path, human_bytes(zip_path.stat().st_size))
    return zip_path


def download_files_one_by_one(paths: list[Path], start_index: int = 0, end_index: int | None = None, sleep_seconds: int = 2):
    selected = paths[start_index:end_index]
    for i, path in enumerate(selected, start=start_index):
        print(f'Downloading index={i}: {path.name} ({human_bytes(path.stat().st_size)})')
        files.download(str(path))
        time.sleep(sleep_seconds)

In [ ]:
@dataclass
class V4Config:
    adaptive_root: str = '/content/drive/MyDrive/recsys/data/gold/adaptive/v1'
    base_gold_root: str = '/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab'
    output_dir: str = '/content/recsys_work/artifacts/adaptive/v4_sequence_retrieval'
    seed: int = 42
    smoke_mode: bool = False
    max_train_examples: int | None = None
    max_validation_examples: int | None = None
    max_item_pool_examples: int | None = None
    batch_size: int = 2048
    eval_batch_size: int = 2048
    epochs: int = 16
    learning_rate: float = 6e-4
    weight_decay: float = 1e-5
    retrieval_dim: int = 128
    user_hidden_dims: tuple[int, ...] = (768, 384)
    item_hidden_dims: tuple[int, ...] = (768, 384)
    transformer_dim: int = 256
    transformer_heads: int = 4
    transformer_layers: int = 2
    transformer_dropout: float = 0.1
    dropout: float = 0.12
    temperature: float = 0.05
    sequence_max_len: int = 30
    negative_sample_count: int = 8192
    popular_negative_fraction: float = 0.7
    item_popularity_column: str = 'item_hist_events'
    item_popularity_alpha: float = 0.75
    eval_top_ks: tuple[int, ...] = (10, 50, 100)
    candidate_eval_user_limit: int | None = None
    candidate_eval_item_limit: int | None = None
    early_stopping_patience: int = 4
    early_stopping_metric: str = 'ndcg@50'
    save_large_cache_to_drive: bool = False
    local_cache_dir: str = '/content/recsys_work/adaptive_cache/v4'
    # Serving-time calibration grid. Keeps raw V4 metrics, then reports the best
    # V4 + item-popularity blend on the same validation candidate set as V3.
    serving_popularity_blend_grid: tuple[float, ...] = (0.0, 0.02, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35)

CONFIG = V4Config()
if CONFIG.smoke_mode:
    CONFIG.max_train_examples = 100_000
    CONFIG.max_validation_examples = 30_000
    CONFIG.max_item_pool_examples = 300_000
    CONFIG.batch_size = 512
    CONFIG.eval_batch_size = 512
    CONFIG.epochs = 1
    CONFIG.negative_sample_count = 1024
    CONFIG.candidate_eval_user_limit = 200
    CONFIG.candidate_eval_item_limit = 100_000

ADAPTIVE_ROOT = Path(CONFIG.adaptive_root)
BASE_GOLD_ROOT = Path(CONFIG.base_gold_root)
OUTPUT_DIR = Path(CONFIG.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_DIR = Path(CONFIG.local_cache_dir)
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(CONFIG), indent=2, default=str))

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parquet_files(path: str | Path) -> list[Path]:
    path = Path(path)
    files = sorted(p for p in path.rglob('*.parquet') if p.is_file() and not p.name.startswith('.'))
    if not files:
        visible = sorted(x.name for x in path.iterdir())[:30] if path.exists() and path.is_dir() else []
        raise FileNotFoundError(f'No parquet files found under {path}. Visible entries: {visible}')
    return files


def read_parquet(path: str | Path, columns=None, max_rows: int | None = None, filter_expr=None) -> pd.DataFrame:
    dataset = ds.dataset([str(p) for p in parquet_files(path)], format='parquet')
    if columns:
        missing = sorted(set(columns) - set(dataset.schema.names))
        if missing:
            raise ValueError(f'{path} missing columns: {missing}')
    if max_rows is None:
        return dataset.to_table(columns=columns, filter=filter_expr).to_pandas()
    return dataset.head(max_rows, columns=columns, filter=filter_expr).to_pandas()


def save_json(path: str | Path, payload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f:
        json.dump(payload, f, indent=2, sort_keys=True, default=str)


def now() -> str:
    return time.strftime('%Y-%m-%d %H:%M:%S')

set_seed(CONFIG.seed)

In [ ]:
required_paths = {
    'adaptive train': ADAPTIVE_ROOT / 'sequence_examples/train',
    'adaptive validation': ADAPTIVE_ROOT / 'sequence_examples/validation',
    'base vocabularies': BASE_GOLD_ROOT / 'vocabularies',
    'base numeric stats': BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
}
for name, path in required_paths.items():
    ok = path.exists()
    print(f'{name:24s}', ok, path)
    if not ok:
        raise FileNotFoundError(path)

In [ ]:
USER_CATEGORICAL = ('user_active_degree',)
ITEM_CATEGORICAL = ('video_type', 'upload_type')
USER_ID_FEATURES = ('user_id',)
ITEM_ID_FEATURES = ('video_id',)
USER_NUMERIC = (
    'is_lowactive_period','is_live_streamer','is_video_author','follow_user_num','fans_user_num','friend_user_num','register_days',
    *[f'onehot_feat{i}' for i in range(18)],
    'user_hist_events','user_hist_long_view_rate','user_hist_like_rate','user_hist_comment_rate','user_hist_forward_rate','user_hist_follow_rate',
    'user_hist_hate_rate','user_hist_avg_watch_ratio','user_hist_avg_play_time_sec','session_event_index','session_elapsed_sec',
    'session_prior_long_view_rate','session_prior_like_rate','session_prior_hate_rate','session_prior_avg_watch_ratio',
    'session_vs_user_long_view_delta','session_vs_user_watch_ratio_delta','event_hour','event_dayofweek','tab','is_rand'
)
ITEM_NUMERIC = (
    'video_duration_sec','aspect_ratio','visible_status','music_type','upload_age_days_at_event','item_hist_events',
    'item_hist_long_view_rate','item_hist_like_rate','item_hist_hate_rate','item_hist_avg_watch_ratio'
)
SEQUENCE_COLUMNS = (
    'recent_video_ids','recent_target_classes','recent_engagement_strengths','recent_long_views','recent_likes',
    'recent_hates','recent_clicks','recent_time_ms','recent_event_count','seconds_since_last_event'
)
TARGET_COLUMNS = ('target_class','is_positive','is_observed_negative','is_ambiguous','long_view','is_like','is_follow','is_hate','engagement_strength')
BASE_COLUMNS = ('example_id','user_id','video_id','as_of_time','current_time_ms')
INPUT_COLUMNS = list(dict.fromkeys([*BASE_COLUMNS, *TARGET_COLUMNS, *USER_ID_FEATURES, *USER_CATEGORICAL, *USER_NUMERIC, *ITEM_ID_FEATURES, *ITEM_CATEGORICAL, *ITEM_NUMERIC, *SEQUENCE_COLUMNS]))
print('input columns:', len(INPUT_COLUMNS))

In [ ]:
class Vocabulary:
    def __init__(self, name: str, values: Iterable, index: dict | None = None):
        self.name = name
        if index is not None:
            self.index = dict(index)
        else:
            clean = pd.Series(list(values)).dropna().drop_duplicates().tolist()
            try:
                clean = sorted(clean)
            except TypeError:
                clean = sorted(str(x) for x in clean)
            self.index = {value: i + 1 for i, value in enumerate(clean)}
        self.size = max(self.index.values(), default=0) + 1

    def encode_many(self, values) -> np.ndarray:
        return np.asarray([self.index.get(v, 0) for v in values], dtype=np.int64)


def load_parquet_vocabulary(path: Path, feature: str) -> Vocabulary:
    df = read_parquet(path)
    idx_col = f'{feature}_idx'
    if idx_col not in df.columns:
        raise ValueError(f'{path} missing {idx_col}')
    return Vocabulary(feature, [], dict(zip(df[feature].tolist(), df[idx_col].astype(int).tolist())))


class NumericalPreprocessor:
    def __init__(self, stats_path: Path):
        payload = json.loads(stats_path.read_text())
        self.stats = payload.get('features', payload)

    def transform(self, frame: pd.DataFrame, features: tuple[str, ...]) -> np.ndarray:
        out = []
        for feature in features:
            if feature not in frame.columns:
                raise ValueError(f'Missing numeric feature {feature}')
            stats = self.stats.get(feature)
            values = pd.to_numeric(frame[feature], errors='coerce').astype('float32')
            if stats is None:
                clean = np.nan_to_num(values.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            else:
                fill = float(stats.get('mean', 0.0) or 0.0)
                values = values.fillna(fill)
                lo = stats.get('p01', stats.get('min'))
                hi = stats.get('p99', stats.get('max'))
                if lo is not None and hi is not None:
                    values = values.clip(float(lo), float(hi))
                mean = float(stats.get('mean', 0.0) or 0.0)
                std = float(stats.get('stddev', 0.0) or 0.0)
                if std > 1e-6:
                    values = (values - mean) / std
                else:
                    values = values * 0.0
                clean = np.nan_to_num(values.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            out.append(clean)
        return np.stack(out, axis=1).astype('float32') if out else np.zeros((len(frame), 0), dtype='float32')


def build_vocabs(train_frame: pd.DataFrame, item_pool: pd.DataFrame) -> dict[str, Vocabulary]:
    return {
        'user_active_degree': load_parquet_vocabulary(BASE_GOLD_ROOT / 'vocabularies/user_active_degree', 'user_active_degree'),
        'video_type': load_parquet_vocabulary(BASE_GOLD_ROOT / 'vocabularies/video_type', 'video_type'),
        'upload_type': load_parquet_vocabulary(BASE_GOLD_ROOT / 'vocabularies/upload_type', 'upload_type'),
        'user_id': Vocabulary('user_id', train_frame['user_id'].tolist()),
        'video_id': Vocabulary('video_id', item_pool['video_id'].tolist()),
        'event_type': Vocabulary('event_type', ['AMBIGUOUS_WEAK', 'OBSERVED_NEGATIVE', 'STRONG_POSITIVE']),
    }

In [ ]:
def as_sequence(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return []


def as_of_time_ms(frame: pd.DataFrame) -> np.ndarray:
    return (pd.to_datetime(frame['as_of_time']).astype('int64') // 1_000_000).to_numpy(dtype='int64')


def sanitize_point_in_time_sequences(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop any sequence events that are not strictly before the target as_of_time.

    This is intentionally conservative and protects training/evaluation even if the
    adaptive data builder attached a same-second history row ambiguously.
    """
    frame = frame.copy()
    boundaries = as_of_time_ms(frame)
    sequence_cols = [
        'recent_video_ids', 'recent_target_classes', 'recent_engagement_strengths',
        'recent_long_views', 'recent_likes', 'recent_hates', 'recent_clicks', 'recent_time_ms',
    ]
    cleaned = {col: [] for col in sequence_cols if col in frame.columns}
    cleaned_counts = []
    dropped_rows = 0
    dropped_events = 0
    for (_, row), boundary in zip(frame.iterrows(), boundaries):
        times = as_sequence(row.get('recent_time_ms'))
        keep = [i for i, t in enumerate(times) if t is not None and not pd.isna(t) and float(t) < float(boundary)]
        if len(keep) != len(times):
            dropped_rows += 1
            dropped_events += len(times) - len(keep)
        for col in cleaned:
            values = as_sequence(row.get(col))
            if len(values) == len(times):
                cleaned[col].append([values[i] for i in keep])
            else:
                cleaned[col].append(values[-len(keep):] if keep else [])
        cleaned_counts.append(len(keep))
    for col, values in cleaned.items():
        frame[col] = values
    frame['recent_event_count'] = cleaned_counts
    frame['current_time_ms'] = boundaries
    print(f'point-in-time sanitize: rows_with_drops={dropped_rows:,} dropped_events={dropped_events:,}')
    return frame

def load_split(split: str, max_examples: int | None) -> pd.DataFrame:
    filt = ds.field('target_class') == 'STRONG_POSITIVE'
    frame = read_parquet(ADAPTIVE_ROOT / 'sequence_examples' / split, INPUT_COLUMNS, max_examples, filt)
    frame = frame.reset_index(drop=True)
    frame = sanitize_point_in_time_sequences(frame)
    print(split, frame.shape)
    return frame


def load_item_pool(split='train', max_examples: int | None = None) -> pd.DataFrame:
    cols = list(dict.fromkeys([*ITEM_ID_FEATURES, *ITEM_CATEGORICAL, *ITEM_NUMERIC]))
    frame = read_parquet(ADAPTIVE_ROOT / 'sequence_examples' / split, cols, max_examples)
    frame = frame.drop_duplicates('video_id', keep='last').reset_index(drop=True)
    print('item pool', split, frame.shape)
    return frame

train_frame = load_split('train', CONFIG.max_train_examples)
validation_frame = load_split('validation', CONFIG.max_validation_examples)
item_pool = load_item_pool('train', CONFIG.max_item_pool_examples)
validation_item_pool = load_item_pool('validation', CONFIG.candidate_eval_item_limit)

if train_frame.empty or validation_frame.empty:
    raise ValueError('No positive examples loaded. Check adaptive sequence_examples and target_class values.')

In [ ]:
numeric = NumericalPreprocessor(BASE_GOLD_ROOT / 'transforms/numeric_stats.json')
vocabs = build_vocabs(train_frame, item_pool)
print({k: v.size for k, v in vocabs.items()})
save_json(OUTPUT_DIR / 'vocab_summary.json', {k: v.size for k, v in vocabs.items()})

In [ ]:
def pad_numeric_sequence(values, max_len: int, dtype='float32'):
    if not isinstance(values, (list, tuple, np.ndarray)):
        values = []
    values = list(values)[-max_len:]
    out = np.zeros(max_len, dtype=dtype)
    if values:
        arr = np.asarray(values, dtype=dtype)
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        out[-len(arr):] = arr
    return out


def pad_encoded_sequence(values, vocab: Vocabulary | None, max_len: int):
    if not isinstance(values, (list, tuple, np.ndarray)):
        values = []
    values = list(values)[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    if values:
        encoded = vocab.encode_many(values) if vocab is not None else np.asarray(values, dtype=np.int64)
        out[-len(values):] = encoded[-max_len:]
    return out


class AdaptiveDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        return self.frame.iloc[idx].to_dict()


class AdaptiveCollator:
    def __init__(self, vocabs, numeric, max_len: int):
        self.vocabs = vocabs
        self.numeric = numeric
        self.max_len = max_len
    def __call__(self, rows):
        frame = pd.DataFrame(rows)
        return encode_frame(frame, self.vocabs, self.numeric, self.max_len)


def encode_frame(frame: pd.DataFrame, vocabs, numeric, max_len: int) -> dict[str, torch.Tensor]:
    batch = {}
    for feature in USER_ID_FEATURES + USER_CATEGORICAL + ITEM_ID_FEATURES + ITEM_CATEGORICAL:
        batch[feature] = torch.as_tensor(vocabs[feature].encode_many(frame[feature].tolist()), dtype=torch.long)
    batch['user_numeric'] = torch.as_tensor(numeric.transform(frame, USER_NUMERIC), dtype=torch.float32)
    batch['item_numeric'] = torch.as_tensor(numeric.transform(frame, ITEM_NUMERIC), dtype=torch.float32)
    batch['seq_video_id'] = torch.as_tensor(np.stack([pad_encoded_sequence(v, vocabs['video_id'], max_len) for v in frame['recent_video_ids']]), dtype=torch.long)
    batch['seq_event_type'] = torch.as_tensor(np.stack([pad_encoded_sequence(v, vocabs['event_type'], max_len) for v in frame['recent_target_classes']]), dtype=torch.long)
    engagement = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_engagement_strengths']])
    long_view = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_long_views']])
    likes = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_likes']])
    hates = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_hates']])
    clicks = np.stack([pad_numeric_sequence(v, max_len) for v in frame['recent_clicks']])
    seq_signals = np.stack([engagement, long_view, likes, hates, clicks], axis=-1).astype('float32')
    batch['seq_signals'] = torch.as_tensor(seq_signals, dtype=torch.float32)
    current_ms = pd.to_numeric(frame['current_time_ms'], errors='coerce').fillna(0).to_numpy(dtype='float64')
    recent_ms = np.stack([pad_numeric_sequence(v, max_len, dtype='float64') for v in frame['recent_time_ms']])
    age_sec = np.maximum((current_ms[:, None] - recent_ms) / 1000.0, 0.0)
    age_sec[recent_ms <= 0] = 0.0
    age_bucket = np.clip(np.floor(np.log1p(age_sec) / np.log(2.0)), 0, 31).astype('int64')
    batch['seq_time_bucket'] = torch.as_tensor(age_bucket, dtype=torch.long)
    batch['seq_padding_mask'] = batch['seq_video_id'].eq(0)
    batch['label'] = torch.arange(len(frame), dtype=torch.long)
    batch['video_id_raw'] = torch.as_tensor(pd.to_numeric(frame['video_id'], errors='coerce').fillna(0).to_numpy(dtype='int64'))
    return batch


def move_batch(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

In [ ]:
def mlp(input_dim, hidden_dims, output_dim, dropout):
    layers = []
    prev = input_dim
    for hidden in hidden_dims:
        layers += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(dropout)]
        prev = hidden
    layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)


class V4TransformerRetrieval(nn.Module):
    def __init__(self, vocab_sizes: dict[str, int], config: V4Config):
        super().__init__()
        self.config = config
        self.user_id_emb = nn.Embedding(vocab_sizes['user_id'], 8, padding_idx=0)
        self.user_active_emb = nn.Embedding(vocab_sizes['user_active_degree'], 4, padding_idx=0)
        self.video_id_emb = nn.Embedding(vocab_sizes['video_id'], 16, padding_idx=0)
        self.video_type_emb = nn.Embedding(vocab_sizes['video_type'], 3, padding_idx=0)
        self.upload_type_emb = nn.Embedding(vocab_sizes['upload_type'], 8, padding_idx=0)
        self.seq_video_emb = nn.Embedding(vocab_sizes['video_id'], config.transformer_dim, padding_idx=0)
        self.seq_event_emb = nn.Embedding(vocab_sizes['event_type'], 32, padding_idx=0)
        self.seq_event_proj = nn.Linear(32, config.transformer_dim)
        self.seq_signal_proj = nn.Linear(5, config.transformer_dim)
        self.seq_time_emb = nn.Embedding(32, config.transformer_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=config.transformer_dim,
            nhead=config.transformer_heads,
            dim_feedforward=config.transformer_dim * 4,
            dropout=config.transformer_dropout,
            batch_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=config.transformer_layers)
        self.short_proj = nn.Sequential(nn.LayerNorm(config.transformer_dim), nn.Linear(config.transformer_dim, config.retrieval_dim), nn.ReLU())
        user_input = 8 + 4 + len(USER_NUMERIC)
        item_input = 16 + 3 + 8 + len(ITEM_NUMERIC)
        self.long_user = mlp(user_input, config.user_hidden_dims, config.retrieval_dim, config.dropout)
        self.item_tower = mlp(item_input, config.item_hidden_dims, config.retrieval_dim, config.dropout)
        self.gate = nn.Sequential(nn.Linear(config.retrieval_dim * 2, config.retrieval_dim), nn.Sigmoid())
        self.temperature = config.temperature

    def encode_sequence(self, batch):
        token = self.seq_video_emb(batch['seq_video_id'])
        token = token + self.seq_event_proj(self.seq_event_emb(batch['seq_event_type']))
        token = token + self.seq_signal_proj(batch['seq_signals'])
        token = token + self.seq_time_emb(batch['seq_time_bucket'].clamp(0, 31))
        mask = batch['seq_padding_mask']
        encoded = self.transformer(token, src_key_padding_mask=mask)
        valid = (~mask).float().unsqueeze(-1)
        pooled = (encoded * valid).sum(1) / valid.sum(1).clamp_min(1.0)
        return self.short_proj(pooled)

    def encode_user(self, batch):
        long_features = torch.cat([
            self.user_id_emb(batch['user_id']),
            self.user_active_emb(batch['user_active_degree']),
            batch['user_numeric'],
        ], dim=1)
        long_repr = self.long_user(long_features)
        short_repr = self.encode_sequence(batch)
        gate = self.gate(torch.cat([long_repr, short_repr], dim=1))
        fused = gate * short_repr + (1.0 - gate) * long_repr
        return F.normalize(fused, dim=1)

    def encode_item(self, batch):
        item_features = torch.cat([
            self.video_id_emb(batch['video_id']),
            self.video_type_emb(batch['video_type']),
            self.upload_type_emb(batch['upload_type']),
            batch['item_numeric'],
        ], dim=1)
        return F.normalize(self.item_tower(item_features), dim=1)

    def forward(self, batch, negative_batch=None):
        user_emb = self.encode_user(batch)
        pos_item_emb = self.encode_item(batch)
        item_embs = [pos_item_emb]
        if negative_batch is not None:
            item_embs.append(self.encode_item(negative_batch))
        all_item_emb = torch.cat(item_embs, dim=0)
        logits = user_emb @ all_item_emb.T / self.temperature
        return logits, user_emb, pos_item_emb

In [ ]:
# Point-in-time check: every sequence timestamp must be strictly earlier than target as_of_time.
def leakage_check(frame: pd.DataFrame, max_rows: int = 200_000):
    sample = frame.head(max_rows)
    boundaries = as_of_time_ms(sample)
    violations = 0
    checked = 0
    for (_, row), boundary in zip(sample.iterrows(), boundaries):
        times = as_sequence(row.get('recent_time_ms'))
        for t in times:
            if t is None or pd.isna(t):
                continue
            checked += 1
            if float(t) >= float(boundary):
                violations += 1
                break
    print('sequence timestamps checked:', checked, 'violating rows:', violations)
    if violations:
        raise AssertionError(f'Point-in-time leakage detected in {violations} rows')

leakage_check(train_frame)
leakage_check(validation_frame)

In [ ]:
vocab_sizes = {k: v.size for k, v in vocabs.items()}
model = V4TransformerRetrieval(vocab_sizes, CONFIG).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
loss_fn = nn.CrossEntropyLoss()
collator = AdaptiveCollator(vocabs, numeric, CONFIG.sequence_max_len)
train_loader = DataLoader(AdaptiveDataset(train_frame), batch_size=CONFIG.batch_size, shuffle=True, collate_fn=collator, num_workers=0)
validation_loader = DataLoader(AdaptiveDataset(validation_frame), batch_size=CONFIG.eval_batch_size, shuffle=False, collate_fn=collator, num_workers=0)
print('params:', sum(p.numel() for p in model.parameters()))

In [ ]:
negative_rng = np.random.default_rng(CONFIG.seed)
popularity = pd.to_numeric(item_pool.get(CONFIG.item_popularity_column, pd.Series(np.ones(len(item_pool)))), errors='coerce').fillna(0).to_numpy('float64')
weights = np.power(np.maximum(popularity, 0.0) + 1.0, CONFIG.item_popularity_alpha)
popularity_probs = weights / weights.sum() if weights.sum() > 0 else np.full(len(item_pool), 1 / len(item_pool))


def sample_negative_frame(count: int) -> pd.DataFrame:
    popular_count = int(round(count * CONFIG.popular_negative_fraction))
    random_count = count - popular_count
    parts = []
    if popular_count:
        parts.append(negative_rng.choice(len(item_pool), size=popular_count, replace=True, p=popularity_probs))
    if random_count:
        parts.append(negative_rng.integers(0, len(item_pool), size=random_count))
    idx = np.concatenate(parts) if parts else np.array([], dtype=np.int64)
    return item_pool.iloc[idx].reset_index(drop=True)


def make_negative_batch(count: int):
    neg_frame = sample_negative_frame(count)
    encoded = encode_frame(neg_frame.assign(**minimal_user_stub(len(neg_frame))), vocabs, numeric, CONFIG.sequence_max_len)
    return move_batch(encoded, DEVICE)


def minimal_user_stub(n: int) -> dict:
    stub = {c: 0 for c in USER_NUMERIC}
    stub.update({
        'user_id': train_frame['user_id'].iloc[0],
        'user_active_degree': train_frame['user_active_degree'].iloc[0],
        'recent_video_ids': [[] for _ in range(n)],
        'recent_target_classes': [[] for _ in range(n)],
        'recent_engagement_strengths': [[] for _ in range(n)],
        'recent_long_views': [[] for _ in range(n)],
        'recent_likes': [[] for _ in range(n)],
        'recent_hates': [[] for _ in range(n)],
        'recent_clicks': [[] for _ in range(n)],
        'recent_time_ms': [[] for _ in range(n)],
        'current_time_ms': 0,
    })
    return stub

In [ ]:
def evaluate_in_batch(loader, max_batches: int | None = 50):
    model.eval()
    totals = {'loss': 0.0, 'batches': 0}
    ks = CONFIG.eval_top_ks
    hits = {k: 0 for k in ks}
    ndcg = {k: 0.0 for k in ks}
    with torch.no_grad():
        for i, raw in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            batch = move_batch(raw, DEVICE)
            logits, _, _ = model(batch)
            logits = torch.nan_to_num(logits, nan=-1e4, posinf=1e4, neginf=-1e4)
            labels = torch.arange(logits.shape[0], device=DEVICE)
            loss = loss_fn(logits, labels)
            if not torch.isfinite(loss):
                loss = torch.zeros((), device=DEVICE)
            ranks = torch.argsort(logits, dim=1, descending=True)
            for k in ks:
                top = ranks[:, :k]
                match = top.eq(labels[:, None])
                hits[k] += match.any(dim=1).sum().item()
                pos = match.float().argmax(dim=1) + 1
                found = match.any(dim=1)
                ndcg[k] += (found.float() / torch.log2(pos.float() + 1.0)).sum().item()
            totals['loss'] += loss.item()
            totals['batches'] += 1
    n = totals['batches'] * CONFIG.eval_batch_size
    out = {'loss': totals['loss'] / max(totals['batches'], 1)}
    for k in ks:
        out[f'hitrate@{k}'] = hits[k] / max(n, 1)
        out[f'ndcg@{k}'] = ndcg[k] / max(n, 1)
    return out

history = []
best_metric = -1.0
best_epoch = None
bad_epochs = 0
for epoch in range(1, CONFIG.epochs + 1):
    model.train()
    running = 0.0
    seen = 0
    start = time.time()
    for step, raw in enumerate(train_loader, start=1):
        batch = move_batch(raw, DEVICE)
        neg = make_negative_batch(CONFIG.negative_sample_count) if CONFIG.negative_sample_count > 0 else None
        logits, _, _ = model(batch, neg)
        logits = torch.nan_to_num(logits, nan=-1e4, posinf=1e4, neginf=-1e4)
        labels = torch.arange(logits.shape[0], device=DEVICE)
        loss = loss_fn(logits, labels)
        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite V4 loss detected after logit sanitization')
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running += loss.item() * logits.shape[0]
        seen += logits.shape[0]
        if step % 50 == 0:
            print(f'epoch={epoch} step={step} train_loss={running/max(seen,1):.4f}', flush=True)
    val = evaluate_in_batch(validation_loader)
    row = {'epoch': epoch, 'train_loss': running / max(seen, 1), **{f'validation_{k}': v for k, v in val.items()}, 'elapsed_min': (time.time() - start) / 60}
    history.append(row)
    print(row, flush=True)
    metric = val.get(CONFIG.early_stopping_metric, val.get('ndcg@50', -1.0))
    if metric > best_metric:
        best_metric = metric
        best_epoch = epoch
        bad_epochs = 0
        torch.save({'model_state_dict': model.state_dict(), 'config': asdict(CONFIG), 'vocab_sizes': vocab_sizes}, OUTPUT_DIR / 'best_model.pt')
    else:
        bad_epochs += 1
        if bad_epochs >= CONFIG.early_stopping_patience:
            print('early stopping')
            break

pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
print('best_epoch:', best_epoch, 'best_metric:', best_metric)

## Export V4 Checkpoint Before V5


In [ ]:
# Persist V4 training metadata before exporting. Candidate evaluation is intentionally skipped here.
save_json(OUTPUT_DIR / 'config.json', asdict(CONFIG))
save_json(OUTPUT_DIR / 'metrics.json', {
    'stage': 'v4_train_only',
    'best_epoch': best_epoch,
    'best_metric': best_metric,
    'history': history,
    'note': 'Candidate retrieval evaluation was skipped in this checkpoint-first notebook.',
})

V4_ARTIFACT_DIR = OUTPUT_DIR
V4_CHECKPOINT_PATH = V4_ARTIFACT_DIR / 'best_model.pt'
print('V4 checkpoint:', V4_CHECKPOINT_PATH, V4_CHECKPOINT_PATH.exists())
if not V4_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing V4 checkpoint: {V4_CHECKPOINT_PATH}')
print('checkpoint size:', human_bytes(V4_CHECKPOINT_PATH.stat().st_size))

v4_report_zip = export_report_bundle('v4_train_only', [V4_ARTIFACT_DIR])
if v4_report_zip is not None:
    files.download(str(v4_report_zip))

v4_files_to_download = split_checkpoint(V4_CHECKPOINT_PATH, CHECKPOINT_EXPORT_ROOT / 'v4', CHECKPOINT_CHUNK_SIZE_MB)
print('V4 checkpoint export files:')
for i, path in enumerate(v4_files_to_download):
    print(i, path.name, human_bytes(path.stat().st_size))

In [ ]:
def download_range(paths, start, end=None):
    print('download range:', start, end)
    download_files_one_by_one(paths, start, end, sleep_seconds=2)

### Download V4 parts 0-4


In [ ]:
download_range(v4_files_to_download, 0, 4)

### Download V4 parts 4-8


In [ ]:
download_range(v4_files_to_download, 4, 8)

### Download V4 parts 8-12


In [ ]:
download_range(v4_files_to_download, 8, 12)

### Download V4 remaining parts + manifest


In [ ]:
download_range(v4_files_to_download, 12, None)

---

# V5 Candidate Ranker


## Goal

Train V5 as a true second-stage ranker on candidates retrieved by V4.

Controlled comparison:

```text
V4 = same candidate set ordered by V4 retrieval score
V5 = same candidate set reranked by V5 rank score
```

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import importlib.util, subprocess, sys
required = ['pyarrow', 'torch', 'pandas', 'numpy']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('packages ready')

In [ ]:
from __future__ import annotations
from dataclasses import asdict, dataclass
from pathlib import Path
import json, math, random, time
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
from torch import nn
from torch.nn import functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
@dataclass
class V5Config:
    adaptive_root: str = '/content/drive/MyDrive/recsys/data/gold/adaptive/v1'
    base_gold_root: str = '/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab'
    v4_dir: str = '/content/recsys_work/artifacts/adaptive/v4_sequence_retrieval'
    output_dir: str = '/content/recsys_work/artifacts/adaptive/v5_adaptive_ranker'
    seed: int = 43
    smoke_mode: bool = False
    max_train_queries: int | None = 160_000
    max_validation_queries: int | None = 50_000
    max_item_pool_examples: int | None = None
    retrieval_candidates_per_query: int = 1000
    train_candidates_per_query: int = 96
    batch_query_size: int = 96
    epochs: int = 10
    learning_rate: float = 5e-4
    weight_decay: float = 1e-5
    hidden_dims: tuple[int, ...] = (1024, 512, 256)
    dropout: float = 0.15
    eval_top_ks: tuple[int, ...] = (10, 50, 100)
    save_large_cache_to_drive: bool = False
    local_cache_dir: str = '/content/recsys_work/adaptive_cache/v5'
    # Train with injected positives + hard negatives. Evaluate on validation item pool,
    # not train item pool, so validation true videos can be present.
    positive_loss_weight: float | None = None

CONFIG = V5Config()
if CONFIG.smoke_mode:
    CONFIG.max_train_queries = 2_000
    CONFIG.max_validation_queries = 500
    CONFIG.max_item_pool_examples = 100_000
    CONFIG.retrieval_candidates_per_query = 300
    CONFIG.train_candidates_per_query = 64
    CONFIG.epochs = 1
OUTPUT_DIR = Path(CONFIG.output_dir); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_DIR = Path(CONFIG.local_cache_dir); LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ADAPTIVE_ROOT = Path(CONFIG.adaptive_root); BASE_GOLD_ROOT = Path(CONFIG.base_gold_root); V4_DIR = Path(CONFIG.v4_dir)
print(json.dumps(asdict(CONFIG), indent=2, default=str))

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(CONFIG.seed)

def parquet_files(path):
    path = Path(path); files = sorted(p for p in path.rglob('*.parquet') if p.is_file() and not p.name.startswith('.'))
    if not files: raise FileNotFoundError(path)
    return files

def read_parquet(path, columns=None, max_rows=None, filter_expr=None):
    dataset = ds.dataset([str(p) for p in parquet_files(path)], format='parquet')
    if max_rows is None: return dataset.to_table(columns=columns, filter=filter_expr).to_pandas()
    return dataset.head(max_rows, columns=columns, filter=filter_expr).to_pandas()

def save_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f: json.dump(payload, f, indent=2, sort_keys=True, default=str)

USER_CATEGORICAL=('user_active_degree',); ITEM_CATEGORICAL=('video_type','upload_type'); USER_ID_FEATURES=('user_id',); ITEM_ID_FEATURES=('video_id',)
USER_NUMERIC=('is_lowactive_period','is_live_streamer','is_video_author','follow_user_num','fans_user_num','friend_user_num','register_days',*[f'onehot_feat{i}' for i in range(18)],'user_hist_events','user_hist_long_view_rate','user_hist_like_rate','user_hist_comment_rate','user_hist_forward_rate','user_hist_follow_rate','user_hist_hate_rate','user_hist_avg_watch_ratio','user_hist_avg_play_time_sec','session_event_index','session_elapsed_sec','session_prior_long_view_rate','session_prior_like_rate','session_prior_hate_rate','session_prior_avg_watch_ratio','session_vs_user_long_view_delta','session_vs_user_watch_ratio_delta','event_hour','event_dayofweek','tab','is_rand')
ITEM_NUMERIC=('video_duration_sec','aspect_ratio','visible_status','music_type','upload_age_days_at_event','item_hist_events','item_hist_long_view_rate','item_hist_like_rate','item_hist_hate_rate','item_hist_avg_watch_ratio')
SEQUENCE_COLUMNS=('recent_video_ids','recent_target_classes','recent_engagement_strengths','recent_long_views','recent_likes','recent_hates','recent_clicks','recent_time_ms','recent_event_count','seconds_since_last_event')
TARGET_COLUMNS=('target_class','is_positive','is_observed_negative','is_ambiguous','long_view','is_like','is_follow','is_hate','engagement_strength')
BASE_COLUMNS=('example_id','user_id','video_id','as_of_time','current_time_ms')
INPUT_COLUMNS=list(dict.fromkeys([*BASE_COLUMNS,*TARGET_COLUMNS,*USER_ID_FEATURES,*USER_CATEGORICAL,*USER_NUMERIC,*ITEM_ID_FEATURES,*ITEM_CATEGORICAL,*ITEM_NUMERIC,*SEQUENCE_COLUMNS]))

In [ ]:
def as_sequence(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return []


def as_of_time_ms(frame: pd.DataFrame) -> np.ndarray:
    return (pd.to_datetime(frame['as_of_time']).astype('int64') // 1_000_000).to_numpy(dtype='int64')


def sanitize_point_in_time_sequences(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop any sequence events that are not strictly before the target as_of_time.

    This is intentionally conservative and protects training/evaluation even if the
    adaptive data builder attached a same-second history row ambiguously.
    """
    frame = frame.copy()
    boundaries = as_of_time_ms(frame)
    sequence_cols = [
        'recent_video_ids', 'recent_target_classes', 'recent_engagement_strengths',
        'recent_long_views', 'recent_likes', 'recent_hates', 'recent_clicks', 'recent_time_ms',
    ]
    cleaned = {col: [] for col in sequence_cols if col in frame.columns}
    cleaned_counts = []
    dropped_rows = 0
    dropped_events = 0
    for (_, row), boundary in zip(frame.iterrows(), boundaries):
        times = as_sequence(row.get('recent_time_ms'))
        keep = [i for i, t in enumerate(times) if t is not None and not pd.isna(t) and float(t) < float(boundary)]
        if len(keep) != len(times):
            dropped_rows += 1
            dropped_events += len(times) - len(keep)
        for col in cleaned:
            values = as_sequence(row.get(col))
            if len(values) == len(times):
                cleaned[col].append([values[i] for i in keep])
            else:
                cleaned[col].append(values[-len(keep):] if keep else [])
        cleaned_counts.append(len(keep))
    for col, values in cleaned.items():
        frame[col] = values
    frame['recent_event_count'] = cleaned_counts
    frame['current_time_ms'] = boundaries
    print(f'point-in-time sanitize: rows_with_drops={dropped_rows:,} dropped_events={dropped_events:,}')
    return frame

In [ ]:
class Vocabulary:
    def __init__(self, name, values, index=None):
        self.name=name
        if index is None:
            clean=pd.Series(list(values)).dropna().drop_duplicates().tolist(); clean=sorted(clean)
            self.index={v:i+1 for i,v in enumerate(clean)}
        else: self.index=dict(index)
        self.size=max(self.index.values(), default=0)+1
    def encode_many(self, values): return np.asarray([self.index.get(v,0) for v in values], dtype=np.int64)

def load_parquet_vocabulary(path, feature):
    df=read_parquet(path); return Vocabulary(feature, [], dict(zip(df[feature].tolist(), df[f'{feature}_idx'].astype(int).tolist())))

class NumericalPreprocessor:
    def __init__(self, path): self.stats=json.loads(Path(path).read_text()).get('features', json.loads(Path(path).read_text()))
    def transform(self, frame, features):
        outs=[]
        for feat in features:
            vals=pd.to_numeric(frame[feat], errors='coerce').astype('float32') if feat in frame.columns else pd.Series(np.zeros(len(frame)), dtype='float32')
            st=self.stats.get(feat)
            if st:
                vals=vals.fillna(float(st.get('mean',0) or 0)); lo=st.get('p01',st.get('min')); hi=st.get('p99',st.get('max'))
                if lo is not None and hi is not None: vals=vals.clip(float(lo),float(hi))
                std=float(st.get('stddev',0) or 0); mean=float(st.get('mean',0) or 0); vals=(vals-mean)/std if std>1e-6 else vals*0
            outs.append(np.nan_to_num(vals.to_numpy('float32'), nan=0, posinf=0, neginf=0))
        return np.stack(outs, axis=1).astype('float32') if outs else np.zeros((len(frame),0),'float32')

def build_vocabs(train_frame, item_pool):
    return {'user_active_degree':load_parquet_vocabulary(BASE_GOLD_ROOT/'vocabularies/user_active_degree','user_active_degree'),'video_type':load_parquet_vocabulary(BASE_GOLD_ROOT/'vocabularies/video_type','video_type'),'upload_type':load_parquet_vocabulary(BASE_GOLD_ROOT/'vocabularies/upload_type','upload_type'),'user_id':Vocabulary('user_id',train_frame['user_id'].tolist()),'video_id':Vocabulary('video_id',item_pool['video_id'].tolist()),'event_type':Vocabulary('event_type',['AMBIGUOUS_WEAK','OBSERVED_NEGATIVE','STRONG_POSITIVE'])}

In [ ]:
def pad_num(v,max_len,dtype='float32'):
    v=[] if not isinstance(v,(list,tuple,np.ndarray)) else list(v)[-max_len:]
    out=np.zeros(max_len,dtype=dtype)
    if v:
        arr=np.asarray(v,dtype=dtype)
        arr=np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        out[-len(arr):]=arr
    return out

def pad_enc(v,vocab,max_len):
    v=[] if not isinstance(v,(list,tuple,np.ndarray)) else list(v)[-max_len:]; out=np.zeros(max_len,dtype=np.int64)
    if v: out[-len(v):]=vocab.encode_many(v)[-max_len:]
    return out

def encode_frame(frame, vocabs, numeric, max_len):
    b={}
    for f in USER_ID_FEATURES+USER_CATEGORICAL+ITEM_ID_FEATURES+ITEM_CATEGORICAL: b[f]=torch.as_tensor(vocabs[f].encode_many(frame[f].tolist()), dtype=torch.long)
    b['user_numeric']=torch.as_tensor(numeric.transform(frame, USER_NUMERIC), dtype=torch.float32); b['item_numeric']=torch.as_tensor(numeric.transform(frame, ITEM_NUMERIC), dtype=torch.float32)
    b['seq_video_id']=torch.as_tensor(np.stack([pad_enc(v,vocabs['video_id'],max_len) for v in frame['recent_video_ids']]), dtype=torch.long)
    b['seq_event_type']=torch.as_tensor(np.stack([pad_enc(v,vocabs['event_type'],max_len) for v in frame['recent_target_classes']]), dtype=torch.long)
    sig=np.stack([np.stack([pad_num(v,max_len) for v in frame[c]]) for c in ['recent_engagement_strengths','recent_long_views','recent_likes','recent_hates','recent_clicks']], axis=-1)
    b['seq_signals']=torch.as_tensor(sig, dtype=torch.float32)
    cur=pd.to_numeric(frame['current_time_ms'],errors='coerce').fillna(0).to_numpy('float64'); rec=np.stack([pad_num(v,max_len,'float64') for v in frame['recent_time_ms']])
    age=np.maximum((cur[:,None]-rec)/1000,0); age[rec<=0]=0; b['seq_time_bucket']=torch.as_tensor(np.clip(np.floor(np.log1p(age)/np.log(2)),0,31).astype('int64'))
    b['seq_padding_mask']=b['seq_video_id'].eq(0)
    return b

def move(batch,device): return {k:v.to(device) if torch.is_tensor(v) else v for k,v in batch.items()}

def minimal_user_stub(source_row, n):
    row=source_row
    data={c: row.get(c,0) for c in USER_NUMERIC}
    data.update({'user_id':row['user_id'],'user_active_degree':row.get('user_active_degree'),'recent_video_ids':[row.get('recent_video_ids',[]) for _ in range(n)],'recent_target_classes':[row.get('recent_target_classes',[]) for _ in range(n)],'recent_engagement_strengths':[row.get('recent_engagement_strengths',[]) for _ in range(n)],'recent_long_views':[row.get('recent_long_views',[]) for _ in range(n)],'recent_likes':[row.get('recent_likes',[]) for _ in range(n)],'recent_hates':[row.get('recent_hates',[]) for _ in range(n)],'recent_clicks':[row.get('recent_clicks',[]) for _ in range(n)],'recent_time_ms':[row.get('recent_time_ms',[]) for _ in range(n)],'current_time_ms':row.get('current_time_ms',0)})
    return data

In [ ]:
# V4 model definition must match notebook 12.
def mlp(input_dim, hidden_dims, output_dim, dropout):
    layers=[]; prev=input_dim
    for h in hidden_dims: layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]; prev=h
    layers.append(nn.Linear(prev,output_dim)); return nn.Sequential(*layers)

class V4TransformerRetrieval(nn.Module):
    def __init__(self, vocab_sizes, cfg):
        super().__init__(); self.cfg=cfg; rd=cfg['retrieval_dim']; td=cfg['transformer_dim']
        self.user_id_emb=nn.Embedding(vocab_sizes['user_id'],8,padding_idx=0); self.user_active_emb=nn.Embedding(vocab_sizes['user_active_degree'],4,padding_idx=0)
        self.video_id_emb=nn.Embedding(vocab_sizes['video_id'],16,padding_idx=0); self.video_type_emb=nn.Embedding(vocab_sizes['video_type'],3,padding_idx=0); self.upload_type_emb=nn.Embedding(vocab_sizes['upload_type'],8,padding_idx=0)
        self.seq_video_emb=nn.Embedding(vocab_sizes['video_id'],td,padding_idx=0); self.seq_event_emb=nn.Embedding(vocab_sizes['event_type'],32,padding_idx=0); self.seq_event_proj=nn.Linear(32,td); self.seq_signal_proj=nn.Linear(5,td); self.seq_time_emb=nn.Embedding(32,td)
        layer=nn.TransformerEncoderLayer(td,cfg['transformer_heads'],td*4,cfg['transformer_dropout'],batch_first=True,activation='gelu'); self.transformer=nn.TransformerEncoder(layer,cfg['transformer_layers'])
        self.short_proj=nn.Sequential(nn.LayerNorm(td),nn.Linear(td,rd),nn.ReLU()); self.long_user=mlp(8+4+len(USER_NUMERIC),tuple(cfg['user_hidden_dims']),rd,cfg['dropout']); self.item_tower=mlp(16+3+8+len(ITEM_NUMERIC),tuple(cfg['item_hidden_dims']),rd,cfg['dropout']); self.gate=nn.Sequential(nn.Linear(rd*2,rd),nn.Sigmoid())
    def encode_sequence(self,b):
        tok=self.seq_video_emb(b['seq_video_id'])+self.seq_event_proj(self.seq_event_emb(b['seq_event_type']))+self.seq_signal_proj(b['seq_signals'])+self.seq_time_emb(b['seq_time_bucket'].clamp(0,31)); mask=b['seq_padding_mask']; enc=self.transformer(tok,src_key_padding_mask=mask); valid=(~mask).float().unsqueeze(-1); return self.short_proj((enc*valid).sum(1)/valid.sum(1).clamp_min(1))
    def encode_user(self,b):
        long=self.long_user(torch.cat([self.user_id_emb(b['user_id']),self.user_active_emb(b['user_active_degree']),b['user_numeric']],1)); short=self.encode_sequence(b); g=self.gate(torch.cat([long,short],1)); return F.normalize(g*short+(1-g)*long,dim=1)
    def encode_item(self,b):
        return F.normalize(self.item_tower(torch.cat([self.video_id_emb(b['video_id']),self.video_type_emb(b['video_type']),self.upload_type_emb(b['upload_type']),b['item_numeric']],1)),dim=1)

class V5Ranker(nn.Module):
    def __init__(self, retrieval_dim, hidden_dims, dropout):
        super().__init__(); inp=retrieval_dim*4 + 1; self.net=mlp(inp, hidden_dims, 1, dropout)
    def forward(self, user_emb, item_emb, v4_score):
        user_emb=torch.nan_to_num(user_emb, nan=0.0, posinf=0.0, neginf=0.0)
        item_emb=torch.nan_to_num(item_emb, nan=0.0, posinf=0.0, neginf=0.0)
        v4_score=torch.nan_to_num(v4_score, nan=0.0, posinf=1.0, neginf=-1.0).clamp(-1.0,1.0)
        x=torch.cat([user_emb,item_emb,user_emb*item_emb,torch.abs(user_emb-item_emb),v4_score[:,None]],1)
        x=torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)
        return self.net(x).squeeze(1)

In [ ]:
cols=INPUT_COLUMNS
train_queries=read_parquet(ADAPTIVE_ROOT/'sequence_examples/train', cols, CONFIG.max_train_queries, ds.field('target_class')=='STRONG_POSITIVE').reset_index(drop=True)
train_queries=sanitize_point_in_time_sequences(train_queries)

ckpt_path = V4_DIR / 'best_model.pt'
assert ckpt_path.exists(), f'Missing V4 checkpoint: {ckpt_path}. Run the V4 section first.'
ckpt=torch.load(ckpt_path, map_location=DEVICE)
v4_cfg=ckpt['config']

# Rebuild V4 vocabularies from the same train-side pools used by V4 training.
v4_train_for_vocab=read_parquet(
    ADAPTIVE_ROOT/'sequence_examples/train',
    cols,
    v4_cfg.get('max_train_examples'),
    ds.field('target_class')=='STRONG_POSITIVE',
).reset_index(drop=True)
v4_train_for_vocab=sanitize_point_in_time_sequences(v4_train_for_vocab)
item_cols=list(dict.fromkeys([*ITEM_ID_FEATURES,*ITEM_CATEGORICAL,*ITEM_NUMERIC]))
item_pool=read_parquet(
    ADAPTIVE_ROOT/'sequence_examples/train',
    item_cols,
    v4_cfg.get('max_item_pool_examples'),
).drop_duplicates('video_id', keep='last').reset_index(drop=True)

print('V5 train queries / item pool:', train_queries.shape, item_pool.shape)
numeric=NumericalPreprocessor(BASE_GOLD_ROOT/'transforms/numeric_stats.json')
vocabs=build_vocabs(v4_train_for_vocab, item_pool)
vocab_sizes=ckpt.get('vocab_sizes', {k:v.size for k,v in vocabs.items()})
print('V4 checkpoint vocab sizes:', vocab_sizes)

v4=V4TransformerRetrieval(vocab_sizes, v4_cfg).to(DEVICE)
v4.load_state_dict(ckpt['model_state_dict'])
v4.eval()
for p in v4.parameters():
    p.requires_grad_(False)
ranker=V5Ranker(v4_cfg['retrieval_dim'], CONFIG.hidden_dims, CONFIG.dropout).to(DEVICE)
opt=torch.optim.AdamW(ranker.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
pos_weight_value = CONFIG.positive_loss_weight or float(max(CONFIG.train_candidates_per_query - 1, 1))
loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
print('V5 positive loss weight:', pos_weight_value)

In [ ]:
def encode_items(frame, batch_size=8192):
    embs=[]; ids=[]
    with torch.no_grad():
        for s in range(0,len(frame),batch_size):
            part=frame.iloc[s:s+batch_size].reset_index(drop=True)
            dummy=part.assign(**minimal_user_stub(train_queries.iloc[0].to_dict(), len(part)))
            emb=v4.encode_item(move(encode_frame(dummy,vocabs,numeric,v4_cfg['sequence_max_len']),DEVICE)).cpu().numpy()
            emb=np.nan_to_num(emb.astype('float32'), nan=0.0, posinf=0.0, neginf=0.0)
            embs.append(emb); ids.append(part['video_id'].to_numpy('int64'))
    return np.concatenate(ids), np.vstack(embs).astype('float32')

candidate_pool_name=None
item_ids=np.array([], dtype='int64')
item_emb=np.empty((0,v4_cfg['retrieval_dim']),'float32')

def set_candidate_pool(pool: pd.DataFrame, name: str):
    global candidate_pool_name, item_ids, item_emb, item_index
    candidate_pool_name=name
    item_ids,item_emb=encode_items(pool)
    item_index=pd.DataFrame({'video_id':item_ids, 'item_pos':np.arange(len(item_ids))})
    print(f'{name} item embedding matrix:', item_emb.shape)

set_candidate_pool(item_pool, 'train')


def retrieve_candidates(query_batch: pd.DataFrame, top_k: int):
    with torch.no_grad():
        enc=move(encode_frame(query_batch,vocabs,numeric,v4_cfg['sequence_max_len']),DEVICE)
        user_emb=v4.encode_user(enc).cpu().numpy()
    user_emb=np.nan_to_num(user_emb.astype('float32'), nan=0.0, posinf=0.0, neginf=0.0)
    scores=np.nan_to_num(user_emb @ item_emb.T, nan=-1e4, posinf=1e4, neginf=-1e4)
    k=min(top_k, scores.shape[1])
    idx=np.argpartition(-scores, kth=k-1, axis=1)[:,:k]
    vals=np.take_along_axis(scores,idx,axis=1)
    order=np.argsort(-vals,axis=1)
    idx=np.take_along_axis(idx,order,axis=1)
    vals=np.take_along_axis(vals,order,axis=1)
    return user_emb.astype('float32'), idx, vals.astype('float32')


def candidate_training_batch(query_batch: pd.DataFrame):
    user_emb, idx, vals = retrieve_candidates(query_batch, CONFIG.retrieval_candidates_per_query)
    labels=[]; uemb=[]; iemb=[]; scores=[]
    for qi,row in query_batch.reset_index(drop=True).iterrows():
        cand_idx=idx[qi,:CONFIG.train_candidates_per_query].copy()
        cand_scores=vals[qi,:CONFIG.train_candidates_per_query].copy()
        true_vid=int(row['video_id'])
        true_positions=np.where(item_ids==true_vid)[0]
        if len(true_positions):
            true_pos=true_positions[0]
            # Always inject the observed positive into training candidates. This trains
            # the ranker to separate the true target from hard V4 negatives even when
            # retrieval did not naturally place it in the first 96 slots yet.
            if true_vid not in item_ids[cand_idx]:
                cand_idx[-1]=true_pos
                cand_scores[-1]=float(user_emb[qi] @ item_emb[true_pos])
        for ci,sc in zip(cand_idx,cand_scores):
            uemb.append(user_emb[qi]); iemb.append(item_emb[ci]); scores.append(sc); labels.append(1.0 if int(item_ids[ci])==true_vid else 0.0)
    uarr=np.nan_to_num(np.vstack(uemb).astype('float32'), nan=0.0, posinf=0.0, neginf=0.0)
    iarr=np.nan_to_num(np.vstack(iemb).astype('float32'), nan=0.0, posinf=0.0, neginf=0.0)
    sarr=np.nan_to_num(np.asarray(scores,dtype='float32'), nan=0.0, posinf=1.0, neginf=-1.0)
    sarr=np.clip(sarr, -1.0, 1.0)
    yarr=np.asarray(labels,dtype='float32')
    return torch.as_tensor(uarr,device=DEVICE), torch.as_tensor(iarr,device=DEVICE), torch.as_tensor(sarr,device=DEVICE,dtype=torch.float32), torch.as_tensor(yarr,device=DEVICE,dtype=torch.float32)

In [ ]:
history=[]
best_loss=float('inf')
best_epoch=None
for epoch in range(1, CONFIG.epochs+1):
    ranker.train()
    order=np.random.permutation(len(train_queries))
    total=0
    loss_sum=0.0
    for start in range(0,len(order),CONFIG.batch_query_size):
        qb=train_queries.iloc[order[start:start+CONFIG.batch_query_size]].reset_index(drop=True)
        ue,ie,sc,y=candidate_training_batch(qb)
        logits=ranker(ue,ie,sc)
        if not torch.isfinite(logits).all():
            raise RuntimeError('Non-finite V5 logits detected. Check candidate_training_batch sanitization.')
        loss=loss_fn(logits,y)
        if not torch.isfinite(loss):
            print('bad batch diagnostics:', {
                'ue_finite': bool(torch.isfinite(ue).all().item()),
                'ie_finite': bool(torch.isfinite(ie).all().item()),
                'score_finite': bool(torch.isfinite(sc).all().item()),
                'label_mean': float(y.mean().item()),
                'score_min': float(sc.min().item()),
                'score_max': float(sc.max().item()),
            })
            raise RuntimeError('Non-finite V5 loss detected')
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ranker.parameters(),5)
        opt.step()
        loss_sum += loss.item()*len(y)
        total += len(y)
        if (start//CONFIG.batch_query_size+1)%20==0:
            print('epoch',epoch,'batch',start//CONFIG.batch_query_size+1,'loss',loss_sum/max(total,1),flush=True)
    train_loss=loss_sum/max(total,1)
    row={'epoch':epoch, 'train_loss':train_loss}
    history.append(row)
    print(row, flush=True)
    if train_loss < best_loss:
        best_loss=train_loss
        best_epoch=epoch
        torch.save({
            'model_state_dict':ranker.state_dict(),
            'config':asdict(CONFIG),
            'v4_config':v4_cfg,
            'best_epoch':best_epoch,
            'best_train_loss':best_loss,
        }, OUTPUT_DIR/'best_model.pt')
        print('saved V5 best_model.pt:', OUTPUT_DIR/'best_model.pt', flush=True)

pd.DataFrame(history).to_csv(OUTPUT_DIR/'training_history.csv',index=False)
assert (OUTPUT_DIR/'best_model.pt').exists(), f'V5 checkpoint was not saved: {OUTPUT_DIR/"best_model.pt"}'
print('best_epoch:', best_epoch, 'best_train_loss:', best_loss)

## Export V5 Checkpoint


In [ ]:
save_json(OUTPUT_DIR/'config.json', asdict(CONFIG))
save_json(OUTPUT_DIR/'metrics.json', {
    'stage': 'v5_train_only',
    'best_epoch': best_epoch,
    'best_train_loss': best_loss,
    'history': history,
    'note': 'V5 ranking evaluation was skipped in this checkpoint-first notebook.',
})
save_json(OUTPUT_DIR/'v4_candidate_training_manifest.json', {
    'retrieval_candidates_per_query':CONFIG.retrieval_candidates_per_query,
    'train_candidates_per_query':CONFIG.train_candidates_per_query,
    'candidate_source':'V4 exact matrix search over train item pool',
    'positive_injection': True,
})

V5_ARTIFACT_DIR = OUTPUT_DIR
V5_CHECKPOINT_PATH = V5_ARTIFACT_DIR / 'best_model.pt'
print('V5 checkpoint:', V5_CHECKPOINT_PATH, V5_CHECKPOINT_PATH.exists())
if not V5_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing V5 checkpoint: {V5_CHECKPOINT_PATH}')
print('checkpoint size:', human_bytes(V5_CHECKPOINT_PATH.stat().st_size))

v5_report_zip = export_report_bundle('v5_train_only', [V5_ARTIFACT_DIR])
if v5_report_zip is not None:
    files.download(str(v5_report_zip))

v5_files_to_download = split_checkpoint(V5_CHECKPOINT_PATH, CHECKPOINT_EXPORT_ROOT / 'v5', CHECKPOINT_CHUNK_SIZE_MB)
print('V5 checkpoint export files:')
for i, path in enumerate(v5_files_to_download):
    print(i, path.name, human_bytes(path.stat().st_size))

### Download V5 parts 0-4


In [ ]:
download_range(v5_files_to_download, 0, 4)

### Download V5 remaining parts + manifest


In [ ]:
download_range(v5_files_to_download, 4, None)